# Chapter 13 - Decision Trees

Decision trees are one of the most intuitive algorithms in machine learning. They mimic
human reasoning by asking a series of yes/no questions about the features, splitting the
data at each step until reaching a prediction. Despite their simplicity, trees are the
foundation of some of the most powerful algorithms in practice (random forests, gradient
boosting).

**Topics covered:**
- How decision trees work: recursive binary splitting
- Gini impurity vs entropy (information gain) for classification
- MSE reduction for regression trees
- DecisionTreeClassifier and DecisionTreeRegressor in sklearn
- Visualising trees with `plot_tree()` and `export_text()`
- Feature importance (`.feature_importances_`)
- Controlling tree complexity: max_depth, min_samples_split, min_samples_leaf, max_features
- Overfitting in trees (deep trees memorise noise)
- Cost-complexity pruning (`ccp_alpha`)
- Visualising decision boundaries for different depths
- Classification example on `make_moons`, regression on synthetic data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.tree import plot_tree, export_text
from sklearn.datasets import make_moons, make_blobs
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, mean_squared_error

np.random.seed(42)
sns.set_style('whitegrid')
%matplotlib inline

---
## 1. The Core Idea: Recursive Binary Splitting

A decision tree learns by repeatedly splitting the data into two groups. At each node,
the algorithm considers every feature and every possible threshold, then picks the split
that best separates the target values.

For a feature $x_j$ and threshold $t$, the split divides the data into:
- **Left:** all samples where $x_j \leq t$
- **Right:** all samples where $x_j > t$

The algorithm is **greedy**: at each step it chooses the locally optimal split without
looking ahead. This makes it fast but means the resulting tree is not guaranteed to be
globally optimal.

The process continues recursively until a stopping condition is met (e.g., maximum depth
reached, or a node contains fewer than a minimum number of samples).

In [ ]:
# Simple 2D dataset to visualise splitting
X_demo, y_demo = make_blobs(n_samples=150, centers=3, cluster_std=1.0, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, depth in zip(axes, [1, 2, 3]):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_demo, y_demo)

    # Create mesh for decision regions
    x_min, x_max = X_demo[:, 0].min() - 1, X_demo[:, 0].max() + 1
    y_min, y_max = X_demo[:, 1].min() - 1, X_demo[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    Z = tree.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='Set2')
    ax.scatter(X_demo[:, 0], X_demo[:, 1], c=y_demo, cmap='Set2',
               edgecolors='k', s=40)
    ax.set_title(f'Depth = {depth}  |  Accuracy: {tree.score(X_demo, y_demo):.3f}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('Decision tree splits at increasing depth', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Notice how each increase in depth adds more axis-aligned splits. Decision tree boundaries
are always **rectangular** because each split tests a single feature against a threshold.

---
## 2. Splitting Criteria for Classification

How does the tree decide which split is "best"? It needs a measure of **impurity** that
quantifies how mixed the classes are in a node. The two most common measures are:

### Gini Impurity

$$G = 1 - \sum_{k=1}^{K} p_k^2$$

where $p_k$ is the proportion of class $k$ in the node. Gini impurity is 0 when all
samples belong to one class (pure node) and reaches its maximum when classes are evenly
distributed.

### Entropy (Information Gain)

$$H = -\sum_{k=1}^{K} p_k \log_2 p_k$$

Entropy also measures disorder. The **information gain** of a split is the reduction in
entropy from parent to children.

In practice, Gini and entropy produce very similar trees. Gini is slightly faster to
compute (no logarithm) and is sklearn's default.

In [ ]:
# Visualise Gini and Entropy as a function of class proportion (binary case)
p = np.linspace(0.001, 0.999, 200)

gini = 1 - p**2 - (1 - p)**2
entropy = -p * np.log2(p) - (1 - p) * np.log2(1 - p)
misclass = 1 - np.maximum(p, 1 - p)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(p, gini, label='Gini impurity', linewidth=2)
ax.plot(p, entropy / 2, label='Entropy / 2 (scaled for comparison)', linewidth=2)
ax.plot(p, misclass, label='Misclassification error', linewidth=2, linestyle='--')
ax.set_xlabel('Proportion of class 1 ($p$)')
ax.set_ylabel('Impurity')
ax.set_title('Impurity measures for binary classification')
ax.legend()
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

Both Gini and entropy are concave functions that peak at $p = 0.5$ (maximum uncertainty)
and equal zero at $p = 0$ or $p = 1$ (pure nodes). The misclassification error is piecewise
linear and less sensitive to changes near the peak, which is why trees do not use it for
splitting (though it can be used for pruning).

In [ ]:
# Compare trees trained with Gini vs Entropy on the same data
X_moons, y_moons = make_moons(n_samples=300, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X_moons, y_moons, test_size=0.25, random_state=42
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, criterion in zip(axes, ['gini', 'entropy']):
    tree = DecisionTreeClassifier(max_depth=5, criterion=criterion, random_state=42)
    tree.fit(X_train, y_train)

    x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
    y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    Z = tree.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='RdBu',
               edgecolors='k', s=40)
    acc = tree.score(X_test, y_test)
    ax.set_title(f'Criterion: {criterion}  |  Test accuracy: {acc:.3f}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.tight_layout()
plt.show()

---
## 3. MSE Reduction for Regression Trees

For regression, the tree predicts the **mean** of the target values in each leaf. The
splitting criterion minimises the **mean squared error (MSE)** within each child node:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \bar{y})^2$$

At each candidate split, the algorithm computes the weighted sum of MSE in the left and
right children and picks the split that reduces total MSE the most.

In [ ]:
# Generate a noisy sine wave for regression
np.random.seed(42)
X_reg = np.sort(5 * np.random.rand(200, 1), axis=0)
y_reg = np.sin(X_reg).ravel() + 0.3 * np.random.randn(200)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, depth in zip(axes, [2, 5, 15]):
    reg = DecisionTreeRegressor(max_depth=depth, random_state=42)
    reg.fit(X_reg, y_reg)

    X_plot = np.linspace(0, 5, 500).reshape(-1, 1)
    y_pred = reg.predict(X_plot)

    ax.scatter(X_reg, y_reg, s=15, alpha=0.5, label='Training data')
    ax.plot(X_plot, y_pred, color='red', linewidth=2, label='Tree prediction')
    ax.plot(X_plot, np.sin(X_plot), color='green', linewidth=1.5,
            linestyle='--', label='True function')
    ax.set_title(f'Depth = {depth}')
    ax.set_xlabel('$x$')
    ax.set_ylabel('$y$')
    ax.legend(loc='best')

plt.suptitle('Regression tree at different depths', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

At depth 2, the tree is too coarse to capture the sine curve. At depth 15, it overfits
the noise. Depth 5 provides a reasonable approximation, though the step-function nature
of trees is clearly visible.

---
## 4. DecisionTreeClassifier in Scikit-Learn

Let us train a proper classification tree on the moons dataset and examine the API.

In [ ]:
# Train a classification tree
clf = DecisionTreeClassifier(max_depth=4, random_state=42)
clf.fit(X_train, y_train)

print(f"Training accuracy: {clf.score(X_train, y_train):.4f}")
print(f"Test accuracy:     {clf.score(X_test, y_test):.4f}")
print(f"Number of leaves:  {clf.get_n_leaves()}")
print(f"Tree depth:        {clf.get_depth()}")
print(f"Number of features: {clf.n_features_in_}")

---
## 5. Visualising Trees

One of the greatest strengths of decision trees is their **interpretability**. Scikit-learn
provides two ways to inspect a trained tree:
- `plot_tree()` — a graphical rendering
- `export_text()` — a text-based representation

In [ ]:
# Graphical tree visualisation
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(clf,
          feature_names=['$x_1$', '$x_2$'],
          class_names=['Class 0', 'Class 1'],
          filled=True,
          rounded=True,
          fontsize=9,
          ax=ax)
ax.set_title('Decision tree trained on make_moons (max_depth=4)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Text-based tree
text_repr = export_text(clf, feature_names=['x1', 'x2'])
print(text_repr)

Each node shows:
- The **splitting condition** (e.g., `x1 <= 0.42`)
- The **impurity** (Gini by default)
- The **number of samples** that reached this node
- The **class distribution** (`value = [n_class0, n_class1]`)
- The **predicted class** (majority class in this node)

---
## 6. Feature Importance

After training, the `.feature_importances_` attribute tells us how much each feature
contributed to reducing impurity across all splits. Importances are normalised to sum
to 1.0.

A feature used in early (high-level) splits typically has higher importance because it
affects more samples.

In [ ]:
# Feature importance on the moons tree
feature_names = ['$x_1$', '$x_2$']
importances = clf.feature_importances_

fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(feature_names, importances, color=['steelblue', 'coral'])
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Feature importance for make_moons tree')
for i, v in enumerate(importances):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center')
plt.tight_layout()
plt.show()

In [ ]:
# A higher-dimensional example to make feature importance more interesting
from sklearn.datasets import load_wine

wine = load_wine()
tree_wine = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_wine.fit(wine.data, wine.target)

# Sort features by importance
idx = np.argsort(tree_wine.feature_importances_)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(np.array(wine.feature_names)[idx], tree_wine.feature_importances_[idx],
        color='steelblue')
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Feature importance: Decision Tree on Wine dataset')
plt.tight_layout()
plt.show()

**Caveat:** Impurity-based feature importance is biased toward features with many unique
values (high cardinality). Permutation importance is a more reliable alternative and
will be covered in the random forests notebook.

---
## 7. Controlling Tree Complexity

An unrestricted decision tree will keep splitting until every leaf is pure (or contains a
single sample). This leads to severe overfitting. Scikit-learn offers several parameters
to constrain tree growth:

| Parameter | Effect | Typical range |
|-----------|--------|---------------|
| `max_depth` | Maximum number of levels from root to any leaf | 3 - 20 |
| `min_samples_split` | Minimum samples required to split a node | 2 - 50 |
| `min_samples_leaf` | Minimum samples required in each leaf | 1 - 20 |
| `max_features` | Number of features to consider for each split | `'sqrt'`, `'log2'`, int, float |
| `max_leaf_nodes` | Maximum number of leaf nodes | Depends on data size |

In [ ]:
# Effect of max_depth on train vs test accuracy
depths = range(1, 21)
train_scores = []
test_scores = []

for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train, y_train)
    train_scores.append(tree.score(X_train, y_train))
    test_scores.append(tree.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(depths, train_scores, 'o-', label='Training accuracy', linewidth=2)
ax.plot(depths, test_scores, 's-', label='Test accuracy', linewidth=2)
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_title('Effect of max_depth on training vs test accuracy (make_moons)')
ax.legend()
ax.set_xticks(list(depths))
plt.tight_layout()
plt.show()

In [ ]:
# Effect of min_samples_leaf
leaf_values = [1, 2, 5, 10, 20, 30, 50]
train_scores_leaf = []
test_scores_leaf = []

for ml in leaf_values:
    tree = DecisionTreeClassifier(min_samples_leaf=ml, random_state=42)
    tree.fit(X_train, y_train)
    train_scores_leaf.append(tree.score(X_train, y_train))
    test_scores_leaf.append(tree.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(leaf_values, train_scores_leaf, 'o-', label='Training accuracy', linewidth=2)
ax.plot(leaf_values, test_scores_leaf, 's-', label='Test accuracy', linewidth=2)
ax.set_xlabel('min_samples_leaf')
ax.set_ylabel('Accuracy')
ax.set_title('Effect of min_samples_leaf on training vs test accuracy')
ax.legend()
plt.tight_layout()
plt.show()

Both plots illustrate the **bias-variance tradeoff**: unrestricted trees (deep, small
leaves) achieve perfect training accuracy but generalise poorly. Constraining the tree
sacrifices some training performance for better test performance.

---
## 8. Overfitting: Deep Trees Memorise Noise

A fully grown tree will perfectly classify the training data (assuming no duplicate
samples with different labels). But this perfection comes at the cost of learning the
noise rather than the signal.

In [ ]:
# Visualise overfitting with decision boundaries
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, depth in zip(axes, [1, 3, 7, None]):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)

    x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
    y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    Z = tree.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='RdBu',
               edgecolors='k', s=30)
    depth_label = depth if depth is not None else 'None (unlimited)'
    train_acc = tree.score(X_train, y_train)
    test_acc = tree.score(X_test, y_test)
    ax.set_title(f'depth={depth_label}\nTrain: {train_acc:.3f}  Test: {test_acc:.3f}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('Overfitting: deeper trees create increasingly complex boundaries',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

The unlimited tree creates many tiny rectangular regions that fit individual noisy points.
Its training accuracy is 100% but test accuracy drops. This is the classic signature of
overfitting.

---
## 9. Cost-Complexity Pruning (ccp_alpha)

An alternative to pre-pruning (setting max_depth, etc.) is **post-pruning**: grow the
full tree first, then remove branches that do not significantly improve performance.

Scikit-learn implements **minimal cost-complexity pruning**. The idea is to add a penalty
for the number of leaves:

$$R_\alpha(T) = R(T) + \alpha \cdot |T|$$

where $R(T)$ is the training error, $|T|$ is the number of leaves, and $\alpha$ (`ccp_alpha`)
is the complexity parameter. Larger $\alpha$ produces simpler trees.

In [ ]:
# Find the effective alpha values
full_tree = DecisionTreeClassifier(random_state=42)
full_tree.fit(X_train, y_train)

path = full_tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas
impurities = path.impurities

print(f"Number of effective alpha values: {len(ccp_alphas)}")
print(f"Alpha range: [{ccp_alphas.min():.6f}, {ccp_alphas.max():.6f}]")

In [ ]:
# Train trees for each alpha and track accuracy
train_accs = []
test_accs = []
n_leaves = []

for alpha in ccp_alphas:
    tree = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    tree.fit(X_train, y_train)
    train_accs.append(tree.score(X_train, y_train))
    test_accs.append(tree.score(X_test, y_test))
    n_leaves.append(tree.get_n_leaves())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy vs alpha
ax = axes[0]
ax.plot(ccp_alphas, train_accs, 'o-', label='Train', markersize=3)
ax.plot(ccp_alphas, test_accs, 's-', label='Test', markersize=3)
ax.set_xlabel('ccp_alpha')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy vs ccp_alpha')
ax.legend()

# Number of leaves vs alpha
ax = axes[1]
ax.plot(ccp_alphas, n_leaves, 'o-', color='green', markersize=3)
ax.set_xlabel('ccp_alpha')
ax.set_ylabel('Number of leaves')
ax.set_title('Tree complexity vs ccp_alpha')

plt.tight_layout()
plt.show()

In [ ]:
# Use cross-validation to find the best alpha
alpha_scores = []

for alpha in ccp_alphas:
    tree = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    scores = cross_val_score(tree, X_train, y_train, cv=5, scoring='accuracy')
    alpha_scores.append(scores.mean())

best_idx = np.argmax(alpha_scores)
best_alpha = ccp_alphas[best_idx]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(ccp_alphas, alpha_scores, 'o-', markersize=3)
ax.axvline(best_alpha, color='red', linestyle='--',
           label=f'Best alpha = {best_alpha:.4f}')
ax.set_xlabel('ccp_alpha')
ax.set_ylabel('Mean CV accuracy')
ax.set_title('Cross-validated accuracy vs ccp_alpha')
ax.legend()
plt.tight_layout()
plt.show()

# Train final pruned tree
pruned_tree = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=42)
pruned_tree.fit(X_train, y_train)
print(f"Best ccp_alpha:   {best_alpha:.4f}")
print(f"Pruned tree depth: {pruned_tree.get_depth()}")
print(f"Pruned tree leaves: {pruned_tree.get_n_leaves()}")
print(f"Test accuracy:     {pruned_tree.score(X_test, y_test):.4f}")

Cost-complexity pruning effectively searches for the simplest tree that still captures
the important patterns in the data. The cross-validated approach ensures we are not
overfitting to the validation set either.

---
## 10. Decision Boundaries at Different Depths

A comprehensive comparison of how tree depth shapes the decision boundary on the moons
dataset, with both training and test points.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
depths_to_plot = [1, 2, 3, 5, 10, None]

x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))

for ax, depth in zip(axes.ravel(), depths_to_plot):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    Z = tree.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='RdBu',
               edgecolors='k', s=40, marker='s', label='Test')

    depth_label = depth if depth is not None else 'Unlimited'
    test_acc = tree.score(X_test, y_test)
    leaves = tree.get_n_leaves()
    ax.set_title(f'depth={depth_label} | leaves={leaves} | test acc={test_acc:.3f}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('Decision boundaries on test data at various depths', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 11. DecisionTreeRegressor: Practical Example

Let us now look at regression trees in more depth, including the effect of different
hyperparameters.

In [ ]:
# Generate more complex regression data
np.random.seed(42)
X_reg2 = np.sort(np.random.uniform(0, 10, 300)).reshape(-1, 1)
y_reg2 = np.sin(X_reg2).ravel() + 0.5 * np.cos(2 * X_reg2).ravel() + 0.2 * np.random.randn(300)

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X_reg2, y_reg2, test_size=0.25, random_state=42
)

# Compare different min_samples_leaf values
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
X_plot2 = np.linspace(0, 10, 500).reshape(-1, 1)

for ax, msl in zip(axes, [1, 10, 30]):
    reg = DecisionTreeRegressor(min_samples_leaf=msl, random_state=42)
    reg.fit(X_tr_r, y_tr_r)
    y_pred = reg.predict(X_plot2)

    ax.scatter(X_tr_r, y_tr_r, s=15, alpha=0.4, label='Train')
    ax.scatter(X_te_r, y_te_r, s=15, alpha=0.4, marker='s', label='Test')
    ax.plot(X_plot2, y_pred, color='red', linewidth=2, label='Prediction')

    train_mse = mean_squared_error(y_tr_r, reg.predict(X_tr_r))
    test_mse = mean_squared_error(y_te_r, reg.predict(X_te_r))
    ax.set_title(f'min_samples_leaf={msl}\nTrain MSE: {train_mse:.3f}  '
                 f'Test MSE: {test_mse:.3f}')
    ax.set_xlabel('$x$')
    ax.set_ylabel('$y$')
    ax.legend(loc='best', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Visualise the regression tree structure (small tree for readability)
small_reg = DecisionTreeRegressor(max_depth=3, random_state=42)
small_reg.fit(X_tr_r, y_tr_r)

fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(small_reg, feature_names=['x'], filled=True, rounded=True,
          fontsize=9, ax=ax)
ax.set_title('Regression tree (max_depth=3)', fontsize=14)
plt.tight_layout()
plt.show()

In a regression tree, each leaf contains the **mean target value** of the training
samples that ended up in that region. The `value` field in each node shows this mean.

---
## 12. Instability of Decision Trees

A major weakness of single decision trees is their **high variance**. Small changes in
the training data can produce completely different trees. This is because the greedy
splitting algorithm makes locally optimal choices — if the first split changes, the
entire structure downstream changes too.

In [ ]:
# Train trees on different random subsets of the data
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))

for ax, seed in zip(axes, [0, 1, 2, 3]):
    # Draw a slightly different subsample
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(X_train), size=int(0.8 * len(X_train)), replace=False)
    X_sub, y_sub = X_train[idx], y_train[idx]

    tree = DecisionTreeClassifier(max_depth=5, random_state=42)
    tree.fit(X_sub, y_sub)
    Z = tree.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_sub[:, 0], X_sub[:, 1], c=y_sub, cmap='RdBu',
               edgecolors='k', s=30)
    ax.set_title(f'Subsample {seed + 1} ({len(X_sub)} points)')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('Different subsamples produce different trees (high variance)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

This instability is exactly what motivates **ensemble methods** — by averaging many
different trees, we can dramatically reduce variance. That is the topic of the next
notebook.

---
## 13. Summary

| Concept | Key takeaway |
|---------|-------------|
| **Splitting** | Greedy, axis-aligned, based on impurity reduction |
| **Classification criteria** | Gini impurity (default) or entropy; very similar in practice |
| **Regression criterion** | MSE reduction; leaves predict the mean |
| **Overfitting** | Deep trees memorise noise; constrain with max_depth, min_samples_leaf, etc. |
| **Pruning** | `ccp_alpha` penalises tree complexity; use CV to find the best value |
| **Feature importance** | Based on total impurity reduction; biased toward high-cardinality features |
| **Strengths** | Interpretable, handles mixed feature types, no scaling needed, fast |
| **Weaknesses** | High variance, axis-aligned boundaries, prone to overfitting |

Decision trees are powerful building blocks but rarely used alone in practice. Their
real strength emerges when combined into ensembles, which we explore next.

---

**Next up:** [02 - Random Forests](02_random_forests.ipynb) — reducing variance by
averaging many trees grown on bootstrap samples.